<a href="https://colab.research.google.com/github/cidadesdofuturo/Cidades-do-Futuro/blob/main/Matriz_de_Impacto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install python-docx

# =====================================================
# MONTAR GOOGLE DRIVE (COLAB)
# =====================================================
from google.colab import drive
drive.mount('/content/drive')

# =====================================================
# IMPORTS
# =====================================================
import pandas as pd
import unicodedata
import re
from docx import Document
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

# =====================================================
# CONFIGURAÇÕES
# =====================================================
BASE_PATH = "/content/drive/MyDrive/Scripts/Matriz de impacto"

ARQ_INDICADORES = f"{BASE_PATH}/indicadores.xlsx"
ARQ_SOLUCOES   = f"{BASE_PATH}/solucoes.xlsx"

COL_MUNICIPIO = "municipio"

# =====================================================
# NORMALIZAÇÃO
# =====================================================
def normalizar(txt):
    if txt is None or str(txt).strip() == "":
        return ""
    txt = str(txt)
    txt = txt.replace("\u00A0", " ").replace("\u200B", " ")
    txt = unicodedata.normalize("NFKD", txt)
    txt = txt.encode("ascii", "ignore").decode("utf-8")
    txt = re.sub(r"[^a-zA-Z0-9]+", " ", txt)
    txt = " ".join(txt.lower().split())
    return txt.strip()

# =====================================================
# NOME DE ARQUIVO SEGURO
# =====================================================
def nome_arquivo_seguro(txt):
    txt = normalizar(txt)
    return txt.replace(" ", "_")

# =====================================================
# PRIORIDADE
# =====================================================
def definir_prioridade(nota):
    if nota <= 3:
        return "1"
    if nota <= 5:
        return "2"
    return "3"

# =====================================================
# WORD – COR DA CÉLULA
# =====================================================
def set_cell_bg(cell, cor_hex):
    tc = cell._tc
    tcPr = tc.get_or_add_tcPr()
    shd = OxmlElement("w:shd")
    shd.set(qn("w:fill"), cor_hex)
    tcPr.append(shd)

# =====================================================
# ESCOLHER MUNICÍPIO
# =====================================================
def escolher_municipio(df):
    municipios = df[COL_MUNICIPIO].tolist()
    print("\nSelecione o município:\n")
    for i, m in enumerate(municipios):
        print(f"{i} - {m}")

    while True:
        try:
            idx = int(input("\nDigite o número do município: "))
            if 0 <= idx < len(municipios):
                return municipios[idx]
        except:
            pass
        print("Número inválido.")

# =====================================================
# EXTRAIR INDICADORES
# =====================================================
def extrair_indicadores(df, municipio):
    linha = df[df[COL_MUNICIPIO] == municipio].iloc[0]
    indicadores = {}
    cols = list(df.columns)

    for i in range(len(cols) - 1):
        nome = cols[i]
        if nome in ["municipio", ""]:
            continue
        try:
            nota = float(linha[cols[i + 1]])
            indicadores[nome] = nota
        except:
            continue

    return indicadores

# =====================================================
# EXTRAIR TÓPICOS
# =====================================================
def extrair_topicos(df, municipio):
    linha = df[df[COL_MUNICIPIO] == municipio].iloc[0]
    topicos = {}

    for col in df.columns:
        if col in ["municipio", ""]:
            continue
        try:
            nota = float(linha[col])
            topicos[col] = nota
        except:
            continue

    return topicos

# =====================================================
# MAIN
# =====================================================
def main():
    print("Carregando indicadores...")
    df_ind = pd.read_excel(ARQ_INDICADORES)
    df_ind.columns = [normalizar(c) for c in df_ind.columns]

    municipio = escolher_municipio(df_ind)
    print(f"\nMunicípio selecionado: {municipio}")

    indicadores = extrair_indicadores(df_ind, municipio)
    topicos = extrair_topicos(df_ind, municipio)

    print(f"Indicadores pareados: {len(indicadores)}")
    print(f"Tópicos diretos: {len(topicos)}")

    print("\nCarregando soluções...")
    df_sol = pd.read_excel(ARQ_SOLUCOES)
    df_sol.columns = [normalizar(c) for c in df_sol.columns]

    resultados = []

    for _, row in df_sol.iterrows():
        solucao  = row.get("solucao", "")
        resumo   = row.get("versao resumida", "")
        dimensao = row.get("dimensao", "")
        impacto  = row.get("impacto esperado", "")

        notas = []
        indicador_usado = ""

        # INDICADORES
        inds_raw = str(row.get("indicador", ""))
        lista_inds = [
            normalizar(i)
            for i in inds_raw.replace(",", ";").split(";")
            if i.strip()
        ]

        for ind in lista_inds:
            if ind in indicadores:
                notas.append(indicadores[ind])
                indicador_usado = ind

        # TÓPICOS
        if not notas:
            topicos_raw = str(row.get("topicos", ""))
            lista_topicos = [
                normalizar(t)
                for t in topicos_raw.replace(",", ";").split(";")
                if t.strip()
            ]

            for top in lista_topicos:
                if top in topicos:
                    notas.append(topicos[top])
                    indicador_usado = top

        # PRIORIDADE
        if notas:
            pior_nota = min(notas)
            prioridade = definir_prioridade(pior_nota)
        else:
            pior_nota = ""
            prioridade = ""
            indicador_usado = ""

        resultados.append([
            solucao,
            resumo,
            dimensao,
            impacto,
            indicador_usado,
            pior_nota,
            prioridade
        ])

    # ORGANIZAÇÃO
    df_res = pd.DataFrame(
        resultados,
        columns=[
            "solucao",
            "resumo",
            "dimensao",
            "impacto",
            "indicador",
            "nota",
            "prioridade"
        ]
    )

    df_res["prioridade_num"] = pd.to_numeric(df_res["prioridade"], errors="coerce")

    df_res = (
        df_res.sort_values(["solucao", "prioridade_num"])
              .drop_duplicates("solucao")
              .sort_values(["dimensao", "prioridade_num"])
    )

    resultados = df_res[
        ["solucao", "resumo", "dimensao", "impacto", "indicador", "nota", "prioridade"]
    ].values.tolist()

    # WORD
    doc = Document()
    doc.add_heading("Matriz de Priorização das Soluções", 0)
    doc.add_paragraph(f"Município: {municipio}")

    table = doc.add_table(rows=1, cols=7)
    table.style = "Table Grid"

    headers = [
        "Solução",
        "Versão resumida",
        "Dimensão",
        "Impacto esperado",
        "Indicador utilizado",
        "Nota",
        "Prioridade"
    ]

    for i, h in enumerate(headers):
        table.rows[0].cells[i].text = h

    for r in resultados:
        cells = table.add_row().cells
        for i, v in enumerate(r):
            cells[i].text = str(v)

        if r[-1] == "1":
            set_cell_bg(cells[-1], "F4CCCC")
        elif r[-1] == "2":
            set_cell_bg(cells[-1], "FFF2CC")
        elif r[-1] == "3":
            set_cell_bg(cells[-1], "D9EAD3")

    # ✅ NOME DO ARQUIVO COM MUNICÍPIO
    municipio_arquivo = nome_arquivo_seguro(municipio)

    doc.save(f"{BASE_PATH}/{municipio_arquivo}_Matriz_Priorizacao_Solucoes_.docx")

    print("\nDocumento Word gerado com sucesso no Google Drive.")

# =====================================================
# START
# =====================================================
if __name__ == "__main__":
    main()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 4.6 MB/s eta 0:00:00
Mounted at /content/drive
Carregando indicadores...

Selecione o município:

0 - Abadia dos Dourados
1 - Abaeté
2 - Abre Campo
3 - Acaiaca
4 - Açucena
5 - Água Boa
6 - Água Comprida
7 - Aguanil
8 - Águas Formosas
9 - Águas Vermelhas
10 - Aimorés
11 - Aiuruoca
12 - Alagoa
13 - Albertina
14 - Além Paraíba
15 - Alfenas
16 - Alfredo Vasconcelos
17 - Almenara
18 - Alpercata
19 - Alpinópolis
20 - Alterosa
21 - Alto Caparaó
22 - Alto Jequitibá
23 - Alto Rio Doce
24 - Alvarenga
25 - Alvinópolis
26 - Alvorada de Minas
27 - Amparo do Serra
28 - Andradas
29 - Andrelândia
30 - Angelândia
31 - Antônio Carlos
32 - Antônio Dias
33 - Antônio Prado de Minas
34 - Araçaí
35 - Aracitaba
36 - Araçuaí
37 - Araguari
38 - Arantina
39 - Araponga
40 - Araporã
41 - Arapuá
42 - Araújos
43 - Araxá
44 - Arceburgo
45 - Arcos
46 - Areado
47 - Argirita
48 - Aricanduva
49 - Arinos
50 - Astolfo Dutra
51 - Ataléia
52 - Augusto de Lima
53 - Ba